# CampaignForge AI: Kaggle Inference Backend

This notebook is the **AI backend** for the CampaignForge website. It runs on Kaggle and is exposed through ngrok.

```text
CampaignForge Website -> FastAPI Backend -> this notebook (via ngrok)
                                            |- Mistral-Nemo  : text (directions, content)
                                            |- SD-Turbo / RealVisXL : images
                                            '- Gemini : Arabic adaptation + scene helper (fallback)
```

**API endpoints:** `GET /health`, `POST /directions`, `POST /content`, `POST /image` (plus `GET /` page and `POST /generate-campaign`).

**Before you run (Kaggle):**
- Accelerator: GPU (T4 x2)
- Internet: ON
- Add dataset: `campaignforge` (contains `CampaignForge_Marketing_Knowledge_Base.pdf`)
- Add secrets: `NGROK_AUTH_TOKEN`, `GOOGLE_API_KEY`

**Run:** *Run All*, top to bottom. The public ngrok URL is printed in the last section. It changes every time the runtime restarts.

## 1. Setup & Dependencies

In [1]:
!pip install -q \
    transformers \
    accelerate \
    bitsandbytes \
    sentence-transformers \
    faiss-cpu \
    langchain \
    langchain-community \
    langchain-core \
    pydantic \
    fastapi \
    uvicorn \
    pyngrok \
    langchain-huggingface

# PDF loader, image models and background removal


In [2]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

import torch

print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## 2. Configuration

In [3]:
# ---- Text model ----
model_id = "mistralai/Mistral-Nemo-Instruct-2407"

# ---- Knowledge base (RAG) ----
pdf_path = "/kaggle/input/datasets/mohamedkhaled89/campaignforge/CampaignForge_Marketing_Knowledge_Base.pdf"
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

# ---- Image models ----
# True  = photoshoot uses the RealVisXL model (required by the current photoshoot flow)
# False = lighter, but the product-photo photoshoot will not work
ULTRA = True

# ---- Arabic (Gemini adapts the English ad copy) ----
ARABIC_STYLE = "egyptian"                  # "egyptian" or "msa"
GEMINI_TEXT_MODEL = "gemini-flash-lite-latest"

# ---- Server ----
PORT = 8000

## 3. Text Model Loading (Mistral)

In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    use_fast=True
)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/181k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.26M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

[transformers] The tokenizer you are loading from 'mistralai/Mistral-Nemo-Instruct-2407' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


model.safetensors.index.json:   0%|          | 0.00/29.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [5]:
def generate_response(
    messages,
    max_new_tokens=300,
    temperature=0.7,
    top_p=0.9
):

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            use_cache=True
        )

    generated_tokens = outputs[
        0
    ][
        inputs.input_ids.shape[1]:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

## 4. Knowledge Base (RAG)
Loads the marketing PDF, splits it, embeds it and builds the FAISS index used for campaign strategy.

In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

/tmp/ipykernel_203/1676847164.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [8]:
pdf_path='/kaggle/input/datasets/mohamedkhaled89/camppp/CampaignForge_Marketing_Knowledge_Base.pdf'
documents = PyPDFLoader(pdf_path).load()

print("Pages:", len(documents))

Pages: 18


In [9]:
text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(
    documents
)

print("Chunks:", len(chunks))

Chunks: 18


In [10]:
embedding = HuggingFaceEmbeddings(
    model_name=embedding_model_name
)

/tmp/ipykernel_203/3847507528.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
vectordb = FAISS.from_documents(
    chunks,
    embedding
)

print("FAISS vector database created successfully!")

FAISS vector database created successfully!


## 5. Data Models

In [12]:
from pydantic import BaseModel, Field
from enum import Enum
from typing import Optional


class CampaignGoal(str, Enum):
    AWARENESS = "Awareness"
    ENGAGEMENT = "Engagement"
    SALES = "Sales"
    LAUNCH = "Launch"


class BrandTone(str, Enum):
    PROFESSIONAL = "Professional"
    PLAYFUL = "Playful"
    LUXURY = "Luxury"
    BOLD = "Bold"
    FRIENDLY = "Friendly"
    CUSTOM = "Custom"


class UserInput(BaseModel):
    product_name: str = Field(min_length=1)
    product_description: str = Field(min_length=1)
    target_audience: str = Field(min_length=1)

    campaign_goal: CampaignGoal
    brand_tone: BrandTone

    custom_tone: Optional[str] = None
    offer_or_price: Optional[str] = None
    key_differentiator: Optional[str] = None


class ProductUnderstanding(BaseModel):
    features: list[str]
    benefits: list[str]
    pain_points: list[str]
    customer_desires: list[str]
    product_positioning: str


class CampaignStrategy(BaseModel):
    core_message: str
    value_proposition: str
    emotional_angle: str
    cta: str
    content_pillars: list[str]


class CampaignDirection(BaseModel):
    name: str
    strategic_angle: str
    key_message: str
    recommended_cta: str
    content_approach: str


class CampaignDirections(BaseModel):
    product_focused: CampaignDirection
    lifestyle_focused: CampaignDirection
    conversion_focused: CampaignDirection


class PlatformContent(BaseModel):
    hook: str
    caption: str
    cta: str
    hashtags: list[str]
    visual_concept: str


class CampaignContent(BaseModel):
    instagram: PlatformContent
    tiktok: PlatformContent


class CampaignState(BaseModel):
    user_input: UserInput
    product_understanding: ProductUnderstanding
    campaign_strategy: CampaignStrategy
    campaign_directions: Optional[CampaignDirections] = None
    selected_direction: Optional[str] = None
    platform_content: Optional[CampaignContent] = None


class FinalCampaign(BaseModel):
    product_name: str
    campaign_goal: str
    brand_tone: str

    campaign_strategy: CampaignStrategy
    selected_direction: CampaignDirection

    instagram: PlatformContent
    tiktok: PlatformContent

## 6. Text Generation Chains
Product Understanding -> Campaign Strategy -> 3 Strategic Directions -> Instagram/TikTok Content. All use Mistral through `generate_response`.

In [13]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

### 6.1 Product Understanding

In [14]:
def llm_call(prompt_value):

    messages = []

    for message in prompt_value.messages:

        messages.append({
            "role": (
                "system"
                if message.type == "system"
                else "user"
            ),
            "content": message.content
        })

    return generate_response(
        messages,
        max_new_tokens=300,
        temperature=0.3,
        top_p=0.9
    )

In [15]:
llm = RunnableLambda(
    llm_call
)

In [16]:
product_parser = PydanticOutputParser(
    pydantic_object=ProductUnderstanding
)

In [17]:
product_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a marketing product analysis specialist.

Your task is to analyze the user's product information
and target audience.

Extract or infer:
- product features
- customer benefits
- customer pain points
- customer desires
- product positioning

Important rules:
1. Base your analysis primarily on the provided product information.
2. You may make reasonable marketing inferences about pain points and desires.
3. Do not invent specific factual claims about the product.
4. Keep the output concise and useful for downstream campaign strategy generation.
5. Return ONLY the requested structured output.

{format_instructions}"""
    ),
    (
        "human",
        """Analyze this product:

Product name:
{product_name}

Product description:
{product_description}

Target audience:
{target_audience}

Campaign goal:
{campaign_goal}

Brand tone:
{brand_tone}

Offer or price:
{offer_or_price}

Key differentiator:
{key_differentiator}
"""
    )
]).partial(
    format_instructions=product_parser.get_format_instructions()
)

In [18]:
product_chain = (
    product_prompt
    | llm
    | product_parser
)

### 6.2 Campaign Strategy (uses the knowledge base)

In [19]:
strategy_parser = PydanticOutputParser(
    pydantic_object=CampaignStrategy
)

In [20]:
strategy_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a marketing campaign strategy specialist.

Your task is to create a clear campaign strategy using:
1. The user's product information
2. The product understanding analysis
3. Relevant marketing knowledge retrieved from the knowledge base

Create:
- core message
- value proposition
- emotional angle
- call to action
- content pillars

STRICT FACTUAL GROUNDING:

Use ONLY information provided by the user and the retrieved
marketing knowledge.

Do NOT invent:
- statistics
- testimonials
- reviews
- customer experiences
- customer results
- promo codes
- deadlines
- expiration dates
- scarcity
- limited-time claims
- artificial urgency
- guarantees
- product capabilities
- product specifications
- percentages not provided
- health claims not provided
- purchasing mechanisms not provided

CONTENT PILLARS RULE:

Content pillars must be based on the actual product,
target audience, campaign strategy, and provided information.

Do NOT create a "Testimonials" pillar because no testimonials
were provided.

Do NOT create pillars based on hypothetical customer stories,
reviews, or experiences.

Use simple descriptive pillar names such as:
- Automatic hydration tracking
- Hydration reminders
- Reusable product design
- Student lifestyle

OFFER RULE:

If an offer or price is provided, use it EXACTLY as provided.

If the offer is "20% off":
- You may say "20% off"
- You may say "Get 20% off the Smart Water Bottle"
- You MUST NOT create a promo code
- You MUST NOT create a deadline
- You MUST NOT say "today"
- You MUST NOT say "limited-time"
- You MUST NOT imply scarcity
- You MUST NOT create artificial urgency

CTA RULE:

The CTA must be supported by the provided information.

Use the provided offer when appropriate.

Do NOT invent:
- "Shop now"
- "Buy now"
- "Order now"
- "Link in bio"
- "Swipe up"
- promo codes
- deadlines

unless such information is explicitly provided.

Avoid absolute claims such as:
- "never forget again"
- "guaranteed"
- "always"
- "100%"
- "perfect"

unless explicitly supported by the input.

Keep the strategy practical, concise, and useful for
downstream campaign generation.

Return ONLY the requested structured output.

Marketing knowledge:
{marketing_knowledge}

{format_instructions}"""
    ),
    (
        "human",
        """Product information:

Product name:
{product_name}

Product description:
{product_description}

Target audience:
{target_audience}

Campaign goal:
{campaign_goal}

Brand tone:
{brand_tone}

Offer or price:
{offer_or_price}

Key differentiator:
{key_differentiator}


Product understanding:

Features:
{features}

Benefits:
{benefits}

Pain points:
{pain_points}

Customer desires:
{customer_desires}

Product positioning:
{product_positioning}
"""
    )
]).partial(
    format_instructions=strategy_parser.get_format_instructions()
)

### 6.3 Strategic Directions

In [21]:
directions_parser = PydanticOutputParser(
    pydantic_object=CampaignDirections
)

In [22]:
directions_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a marketing campaign creative strategist.

Create three distinct strategic directions for the same campaign.

The three directions must be:

1. Product-Focused
   Focus on product features, capabilities, and functional benefits.

2. Lifestyle-Focused
   Focus on the target audience's lifestyle, problems, desires, and
   how the product fits naturally into their life.

3. Conversion-Focused
   Focus on the campaign goal and the provided offer or price.

STRICT FACTUAL GROUNDING:

You MUST NOT invent or add:
- promo codes
- discount amounts
- deadlines
- limited-time claims
- scarcity
- urgency claims
- testimonials
- reviews
- statistics
- guarantees
- product capabilities
- customer results

The offer or price must be treated literally.

If the offer is "20% off":
- You may say "20% off"
- You may say "Get 20% off"
- You may NOT create a promo code
- You may NOT create a deadline
- You may NOT say "limited-time"
- You may NOT claim scarcity
- You may NOT create artificial urgency

The recommended CTA must use ONLY information provided
in the campaign input and strategy.

All three directions must:
- remain consistent with the same product
- remain consistent with the campaign strategy
- be meaningfully different
- be concise and practical
- avoid unsupported claims

Return ONLY the requested structured output.

{format_instructions}"""
    ),
    (
        "human",
        """Create three campaign directions using the following information.

Product:
{product_name}

Product description:
{product_description}

Target audience:
{target_audience}

Campaign goal:
{campaign_goal}

Brand tone:
{brand_tone}

Offer or price:
{offer_or_price}

Key differentiator:
{key_differentiator}

Product understanding:

Features:
{features}

Benefits:
{benefits}

Pain points:
{pain_points}

Customer desires:
{customer_desires}

Product positioning:
{product_positioning}

Campaign strategy:

Core message:
{core_message}

Value proposition:
{value_proposition}

Emotional angle:
{emotional_angle}

CTA:
{cta}

Content pillars:
{content_pillars}
"""
    )
]).partial(
    format_instructions=directions_parser.get_format_instructions()
)

In [23]:
def directions_llm_call(prompt_value):

    messages = []

    for message in prompt_value.messages:
        messages.append({
            "role": (
                "system"
                if message.type == "system"
                else "user"
            ),
            "content": message.content
        })

    return generate_response(
        messages,
        max_new_tokens=600,
        temperature=0.3,
        top_p=0.9
    )


directions_llm = RunnableLambda(directions_llm_call)

directions_chain = (
    directions_prompt
    | directions_llm
    | directions_parser
)

### 6.4 Instagram / TikTok Content

In [24]:
platform_parser = PydanticOutputParser(
    pydantic_object=CampaignContent
)

In [25]:
platform_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a social media marketing content specialist.

Create social media content for Instagram and TikTok from the
provided campaign information.

IMPORTANT:
The output must contain ONLY information supported by the input.

DO NOT invent ANY information.

For the offer:
- The only provided offer is: {offer_or_price}
- If it is "20% off", write ONLY "20% off"
- NEVER create a promo code
- NEVER create a coupon code
- NEVER write "HYDRATE20" or any other code
- NEVER invent a deadline
- NEVER say "limited-time"
- NEVER create scarcity
- NEVER create artificial urgency

For the product:
- Use only the product features explicitly provided.
- "Reusable" means reusable.
- Do NOT change "reusable" into "100% reusable".
- Do NOT invent specifications, capabilities, results, or statistics.

For calls to action:
- Use the provided CTA or recommended CTA.
- Do NOT invent "Swipe up", "Link in bio", "Buy now",
  "Shop now", or other purchasing mechanisms unless explicitly
  provided in the input.

For claims:
- Do NOT invent health statistics.
- Do NOT invent customer results.
- Do NOT invent testimonials or reviews.
- Do NOT invent scientific or market claims.

Instagram:
- Create an engaging hook.
- Create an Instagram caption.
- Create a CTA.
- Create 3-5 relevant hashtags.
- Every hashtag MUST start with "#".
- Create a visual concept.

TikTok:
- Create a strong short-form video hook.
- Create a conversational caption.
- Create a CTA.
- Create 3-5 relevant hashtags.
- Every hashtag MUST start with "#".
- Create a visual concept.

Instagram and TikTok must be adapted to their platforms,
but must remain consistent with the same selected campaign direction.

Return ONLY the requested structured output.

{format_instructions}"""
    ),
    (
        "human",
        """Create Instagram and TikTok content using the following campaign.

Product:
{product_name}

Product description:
{product_description}

Target audience:
{target_audience}

Campaign goal:
{campaign_goal}

Brand tone:
{brand_tone}

Offer or price:
{offer_or_price}

Key differentiator:
{key_differentiator}

Campaign strategy:

Core message:
{core_message}

Value proposition:
{value_proposition}

Emotional angle:
{emotional_angle}

CTA:
{cta}

Content pillars:
{content_pillars}

Selected campaign direction:

Name:
{direction_name}

Strategic angle:
{strategic_angle}

Key message:
{key_message}

Recommended CTA:
{recommended_cta}

Content approach:
{content_approach}
"""
    )
]).partial(
    format_instructions=platform_parser.get_format_instructions()
)

In [26]:
def platform_llm_call(prompt_value):

    messages = []

    for message in prompt_value.messages:
        messages.append({
            "role": (
                "system"
                if message.type == "system"
                else "user"
            ),
            "content": message.content
        })

    return generate_response(
        messages,
        max_new_tokens=700,
        temperature=0.3,
        top_p=0.9
    )


platform_llm = RunnableLambda(platform_llm_call)

platform_chain = (
    platform_prompt
    | platform_llm
    | platform_parser
)

## 7. Campaign Pipeline Functions

In [27]:
def retrieve_knowledge(user_input, k=3):
    query = f"""
    Marketing strategy for a {user_input.campaign_goal.value} campaign
    targeting {user_input.target_audience}
    with a {user_input.brand_tone.value} brand tone.
    """
    return vectordb.similarity_search(query, k=k)


def format_knowledge_sources(docs, excerpt_chars=240):
    """Turn retrieved chunks into small dicts the UI can show."""
    sources = []
    for doc in docs:
        lines = [l.strip() for l in doc.page_content.splitlines() if l.strip()]
        first = lines[0] if lines else ""
        page = doc.metadata.get("page")
        page_no = page + 1 if isinstance(page, int) else None
        # use the first line as a section label only if it looks like a heading
        is_heading = 3 <= len(first) <= 70 and not first.endswith((".", ",", ";"))
        if is_heading:
            label = first.rstrip(":")
        else:
            label = f"Knowledge base, page {page_no}" if page_no else "Knowledge base"
        text = " ".join(doc.page_content.split())
        sources.append({
            "section": label,
            "page": page_no,
            "excerpt": text[:excerpt_chars] + ("..." if len(text) > excerpt_chars else "")
        })
    return sources


def generate_campaign_strategy(user_input, product_result, docs=None):

    if docs is None:
        docs = retrieve_knowledge(user_input)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = strategy_prompt.invoke({
        "marketing_knowledge": context,

        "product_name": user_input.product_name,
        "product_description": user_input.product_description,
        "target_audience": user_input.target_audience,
        "campaign_goal": user_input.campaign_goal.value,
        "brand_tone": user_input.brand_tone.value,
        "offer_or_price": user_input.offer_or_price or "Not provided",
        "key_differentiator": user_input.key_differentiator or "Not provided",

        "features": product_result.features,
        "benefits": product_result.benefits,
        "pain_points": product_result.pain_points,
        "customer_desires": product_result.customer_desires,
        "product_positioning": product_result.product_positioning
    })

    messages = []

    for message in prompt.messages:

        messages.append({
            "role": (
                "system"
                if message.type == "system"
                else "user"
            ),
            "content": message.content
        })

    response = generate_response(
        messages,
        max_new_tokens=300,
        temperature=0.3,
        top_p=0.9
    )

    return strategy_parser.parse(response)

In [28]:
GOAL_TO_DIRECTION = {
    "Awareness": ("lifestyle_focused", "Lifestyle content builds awareness by showing the product in the audience's daily life."),
    "Engagement": ("lifestyle_focused", "Relatable lifestyle content gets the audience reacting and sharing."),
    "Sales": ("conversion_focused", "Conversion-focused messaging puts the offer and the call to action first."),
    "Launch": ("product_focused", "A launch needs the audience to understand what the product is and does."),
}


def recommend_direction(goal):
    key, why = GOAL_TO_DIRECTION.get(goal, ("lifestyle_focused", ""))
    return {"key": key, "reason": why}


def generate_campaign(user_input):

    # 1. Product Understanding
    product_result = product_chain.invoke({
        "product_name": user_input.product_name,
        "product_description": user_input.product_description,
        "target_audience": user_input.target_audience,
        "campaign_goal": user_input.campaign_goal.value,
        "brand_tone": user_input.brand_tone.value,
        "offer_or_price": user_input.offer_or_price or "Not provided",
        "key_differentiator": user_input.key_differentiator or "Not provided"
    })

    # 2. Campaign Strategy
    docs = retrieve_knowledge(user_input)
    strategy_result = generate_campaign_strategy(
        user_input,
        product_result,
        docs
    )

    # 3. Campaign Directions
    directions_result = directions_chain.invoke({
        "product_name": user_input.product_name,
        "product_description": user_input.product_description,
        "target_audience": user_input.target_audience,
        "campaign_goal": user_input.campaign_goal.value,
        "brand_tone": user_input.brand_tone.value,
        "offer_or_price": user_input.offer_or_price or "Not provided",
        "key_differentiator": user_input.key_differentiator or "Not provided",

        "features": product_result.features,
        "benefits": product_result.benefits,
        "pain_points": product_result.pain_points,
        "customer_desires": product_result.customer_desires,
        "product_positioning": product_result.product_positioning,

        "core_message": strategy_result.core_message,
        "value_proposition": strategy_result.value_proposition,
        "emotional_angle": strategy_result.emotional_angle,
        "cta": strategy_result.cta,
        "content_pillars": strategy_result.content_pillars
    })

    return {
        "user_input": user_input,
        "product_understanding": product_result,
        "campaign_strategy": strategy_result,
        "campaign_directions": directions_result,
        "knowledge_used": format_knowledge_sources(docs)
    }


print("Campaign pipeline is ready.")

Campaign pipeline is ready.


In [29]:
def generate_selected_campaign(campaign_data, direction_choice):

    directions = campaign_data["campaign_directions"]

    if direction_choice == 1:
        selected_direction = directions.product_focused

    elif direction_choice == 2:
        selected_direction = directions.lifestyle_focused

    elif direction_choice == 3:
        selected_direction = directions.conversion_focused

    else:
        raise ValueError("Direction choice must be 1, 2, or 3.")

    platform_result = platform_chain.invoke({
        "product_name": campaign_data["user_input"].product_name,
        "product_description": campaign_data["user_input"].product_description,
        "target_audience": campaign_data["user_input"].target_audience,
        "campaign_goal": campaign_data["user_input"].campaign_goal.value,
        "brand_tone": campaign_data["user_input"].brand_tone.value,
        "offer_or_price": campaign_data["user_input"].offer_or_price or "Not provided",
        "key_differentiator": campaign_data["user_input"].key_differentiator or "Not provided",

        "core_message": campaign_data["campaign_strategy"].core_message,
        "value_proposition": campaign_data["campaign_strategy"].value_proposition,
        "emotional_angle": campaign_data["campaign_strategy"].emotional_angle,
        "cta": campaign_data["campaign_strategy"].cta,
        "content_pillars": campaign_data["campaign_strategy"].content_pillars,

        "direction_name": selected_direction.name,
        "strategic_angle": selected_direction.strategic_angle,
        "key_message": selected_direction.key_message,
        "recommended_cta": selected_direction.recommended_cta,
        "content_approach": selected_direction.content_approach
    })

    return {
        "selected_direction": selected_direction,
        "platform_content": platform_result
    }


print("Selected campaign pipeline is ready.")

Selected campaign pipeline is ready.


In [30]:
def run_campaign(user_input, direction_choice):


    campaign_data = generate_campaign(user_input)

    # Step 2: Select direction and generate
    # platform-specific content
    selected_campaign_data = generate_selected_campaign(
        campaign_data,
        direction_choice
    )

    # Step 3: Build final campaign
    final_campaign = FinalCampaign(
        product_name=campaign_data["user_input"].product_name,
        campaign_goal=campaign_data["user_input"].campaign_goal.value,
        brand_tone=campaign_data["user_input"].brand_tone.value,

        campaign_strategy=campaign_data["campaign_strategy"],

        selected_direction=selected_campaign_data["selected_direction"],

        instagram=selected_campaign_data["platform_content"].instagram,
        tiktok=selected_campaign_data["platform_content"].tiktok
    )

    return final_campaign


print("End-to-end CampaignForge pipeline is ready.")

End-to-end CampaignForge pipeline is ready.


In [31]:
def create_user_input(data):

    return UserInput(
        product_name=data["product_name"],
        product_description=data["product_description"],
        target_audience=data["target_audience"],

        campaign_goal=CampaignGoal(
            data["campaign_goal"]
        ),

        brand_tone=BrandTone(
            data["brand_tone"]
        ),

        custom_tone=data.get("custom_tone"),
        offer_or_price=data.get("offer_or_price"),
        key_differentiator=data.get("key_differentiator")
    )


print("App input converter is ready.")

App input converter is ready.


## 8. Gemini Helpers & Arabic Support
Mistral writes the English ad. For Arabic, Gemini adapts it (Egyptian dialect by default). `_gemini_json` is also used by the product-photo scene helper.

In [32]:
import os, re, json, requests
from pydantic import BaseModel

STYLE_TEXT = {
    "egyptian": "natural, catchy Egyptian colloquial Arabic (العامية المصرية), the way a modern Egyptian brand talks on social media",
    "msa": "simple, friendly Modern Standard Arabic (فصحى مبسطة) that sounds natural on social media",
}
STYLE_EXAMPLE = (
    "خليك في المود بتاعك… وسيب الدوشة علينا.🎧\n"
    "مع SpaceBuds Fit، استمتع بـ Adaptive ANC لحد 45dB وبطارية بتوصل لـ60 ساعة تشغيل، "
    "عشان صوتك المفضل يفضل معاك وقت أطول."
)
BAD_AR = ["عرض لفترة محدودة", "لفترة محدودة", "آخر فرصة", "الرابط في البايو", "الرابط في البيو",
          "اشتري الآن", "اشترِ الآن", "كود خصم", "كوبون"]


def get_google_key():
    key = os.environ.get("GOOGLE_API_KEY")
    if key:
        return key
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GOOGLE_API_KEY")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("GOOGLE_API_KEY")
    except Exception:
        pass
    raise RuntimeError("GOOGLE_API_KEY secret not found. Add it in the notebook Secrets and allow access.")


def _gemini_json(prompt):
    r = requests.post(
        f"https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_TEXT_MODEL}:generateContent",
        headers={"x-goog-api-key": get_google_key(), "Content-Type": "application/json"},
        json={"contents": [{"parts": [{"text": prompt}]}],
              "generationConfig": {"temperature": 0.5, "responseMimeType": "application/json"}},
        timeout=90,
    )
    r.raise_for_status()
    text = r.json()["candidates"][0]["content"]["parts"][0]["text"]
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip()
    return json.loads(text)


def arabic_issues(content, allowed):
    text = " ".join(str(v) for p in content.values() for v in p.values())
    issues = [f"banned phrase: {p}" for p in BAD_AR if p in text and p not in allowed]
    if "wireless" in allowed.lower() and re.search(r"(?<!لا)سلكي", text):
        issues.append("calls a wireless product wired")
    if "لاسر" in text:
        issues.append("typo: write لاسلكية, not لاسرية")
    if re.search(r"(^|[\s\"'])كن\s", text):
        issues.append("uses formal كن: write خليك من الأوائل in Egyptian style")
    return issues


def adapt_to_arabic(english, facts):
    allowed = " ".join(str(v) for v in facts.values())
    base = (
        "You are an expert social-media copywriter. Adapt (do NOT translate word for word) the English "
        "Instagram and TikTok ad copy below into " + STYLE_TEXT[ARABIC_STYLE] + ".\n\n"
        "STYLE EXAMPLE to imitate (tone, rhythm and mixing English terms; it is about a different product):\n"
        + STYLE_EXAMPLE + "\n\nRULES:\n"
        "- Short, catchy, natural. Not formal textbook Arabic.\n"
        "- Stay in ONE dialect: do not mix formal commands such as كن with colloquial endings. Use forms like خليك and جرّب.\n"
        "- Proofread the spelling of every Arabic word before answering.\n"
        "- Pick emojis that match the product (🎧 for earbuds).\n"
        "- Keep the product name and technical terms in Latin letters exactly as in the source.\n"
        "- Use ONLY facts found in PRODUCT FACTS or the English copy. Never invent features, numbers, offers, "
        "promo codes, deadlines, scarcity, 'link in bio' or 'buy now'.\n"
        "- If the product is wireless, never call it wired (use لاسلكية).\n"
        "- If an offer is given, keep it exactly. If not, mention no offer.\n"
        "- Keep each CTA consistent with the English CTA.\n"
        "- Hashtags: Arabic or English, each starts with '#', no spaces (use _), 3 to 5 per platform, "
        "and different lists for Instagram and TikTok.\n"
        "- At most 2 emojis per field.\n"
        "- Leave visual_concept unchanged in English.\n"
        "Return ONLY a JSON object with the keys instagram and tiktok. Each has hook, caption, cta, "
        "hashtags (a list of strings) and visual_concept.\n\n"
        "PRODUCT FACTS:\n" + json.dumps(facts, ensure_ascii=False) +
        "\n\nENGLISH COPY:\n" + json.dumps(english, ensure_ascii=False)
    )
    note, last = "", None
    for _ in range(2):
        try:
            out = _gemini_json(base + note)
            for p in ("instagram", "tiktok"):
                for k in ("hook", "caption", "cta", "hashtags"):
                    assert out[p][k], f"missing {p}.{k}"
                out[p]["hashtags"] = [h if h.startswith("#") else "#" + h.lstrip("#") for h in out[p]["hashtags"]]
                out[p]["visual_concept"] = english[p]["visual_concept"]      # image prompts stay English
            issues = arabic_issues(out, allowed)
            if not issues:
                return out, []
            last, note = (out, issues), "\n\nFIX THESE PROBLEMS FROM YOUR LAST TRY: " + "; ".join(issues)
        except Exception as e:
            last = None if last is None else last
            note = ""
            err = e
    if last:
        return last
    raise err

## 9. Image Models

### 9.1 Photoshoot models (RealVisXL)
Loaded for the product-photo photoshoot. `inpaint_pipe` is kept loaded exactly as in the working version.

In [33]:
# ===== PHOTOSHOOT STEP 1: load the realistic image models (free, open weights) =====
import torch, gc
from diffusers import AutoPipelineForInpainting, StableDiffusionXLImg2ImgPipeline, AutoencoderKL


def _load(cls, name, **kw):
    try:
        return cls.from_pretrained(name, torch_dtype=torch.float16, variant="fp16", **kw)
    except Exception:
        return cls.from_pretrained(name, torch_dtype=torch.float16, **kw)


def _fresh_vae():
    # fp16-safe SDXL VAE (the default one can give black or noisy images in fp16)
    return AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)


gc.collect()
torch.cuda.empty_cache()
free = [torch.cuda.mem_get_info(i)[0] for i in range(torch.cuda.device_count())]
GPU = max(range(len(free)), key=lambda i: free[i])        # the GPU with the most free memory
print("Using GPU", GPU, "with", round(free[GPU] / 1e9, 1), "GB free")

inpaint_pipe = _load(AutoPipelineForInpainting, "diffusers/stable-diffusion-xl-1.0-inpainting-0.1", vae=_fresh_vae())
inpaint_pipe.enable_model_cpu_offload(gpu_id=GPU)

refine_pipe = None
if ULTRA:
    refine_pipe = _load(StableDiffusionXLImg2ImgPipeline, "SG161222/RealVisXL_V4.0", vae=_fresh_vae())
    refine_pipe.enable_model_cpu_offload(gpu_id=GPU)

print("Photoshoot models ready. Ultra mode:", ULTRA)

[transformers] `Siglip2ImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Siglip2ImageProcessor` instead.


Using GPU 0 with 12.9 GB free


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors: reconstructing file:   0%|          |  0.00B /  335MB            

...

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Photoshoot models ready. Ultra mode: True


### 9.2 Background removal (rembg)

In [34]:
from PIL import Image, ImageFilter, ImageChops, ImageOps
from rembg import remove, new_session

try:
    rembg_session = new_session("isnet-general-use")      # cleaner product edges than u2net
except Exception:
    rembg_session = new_session("u2net")

print("rembg ready")

  0%|                                               | 0.00/179M [00:00<?, ?B/s]

rembg ready


### 9.3 Generic ad image model (SD-Turbo)
Used by `/image` when no product photo is uploaded.

In [35]:
from diffusers import AutoPipelineForText2Image

free = [torch.cuda.mem_get_info(i)[0] for i in range(torch.cuda.device_count())]
GPU_SMALL = max(range(len(free)), key=lambda i: free[i])

image_pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo", torch_dtype=torch.float16, variant="fp16"
)
image_pipe.enable_model_cpu_offload(gpu_id=GPU_SMALL)

print("image_pipe ready on GPU", GPU_SMALL)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


model_index.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


image_pipe ready on GPU 0


## 10. Product Photo & Photoshoot
Flow: product photo -> background removal -> scene generation -> product placed in scene -> lighting blend -> real product pixels pasted back on top.

### 10.1 Upload validation

In [36]:
import base64, io, binascii
from fastapi import HTTPException
from PIL import Image, ImageOps
from pydantic import BaseModel

MAX_PHOTO_BYTES = 5 * 1024 * 1024            # decoded size limit: 5 MB
MAX_PHOTO_PIXELS = 25_000_000                # protects memory
ALLOWED_FORMATS = {"PNG", "JPEG", "WEBP"}


class ImageRequest(BaseModel):
    product_name: str
    product_description: str = ""
    visual_concept: str
    platform: str = "instagram"
    product_photo: str | None = None         # data URL (base64) of the real product photo


def load_product_photo(data_url):
    """Decode and validate the uploaded photo. Raises HTTPException(400/413) on problems."""
    try:
        b64 = data_url.split(",", 1)[1] if data_url.startswith("data:") else data_url
        raw = base64.b64decode(b64, validate=True)
    except (binascii.Error, IndexError, ValueError):
        raise HTTPException(400, "The product photo could not be read. Upload a PNG, JPG or WEBP file.")
    if len(raw) > MAX_PHOTO_BYTES:
        raise HTTPException(413, "The product photo is too large. Maximum size is 5 MB.")
    try:
        img = Image.open(io.BytesIO(raw))
        img.verify()
        img = Image.open(io.BytesIO(raw))
    except Exception:
        raise HTTPException(400, "The file is not a valid image. Upload a PNG, JPG or WEBP file.")
    if img.format not in ALLOWED_FORMATS:
        raise HTTPException(400, "Unsupported image type. Upload a PNG, JPG or WEBP file.")
    if img.width * img.height > MAX_PHOTO_PIXELS:
        raise HTTPException(400, "The product photo has too many pixels. Use a smaller image.")
    img = ImageOps.exif_transpose(img).convert("RGBA")
    img.thumbnail((1024, 1024))              # keeps the cut-out fast
    return img

### 10.2 Cut-out, placement and shadow helpers

In [37]:
NEGATIVE = ("text, watermark, logo, caption, extra objects, people, hands, deformed, distorted, "
            "blurry, low quality, cartoon, illustration, painting, oversaturated")


def clean_cutout(rgba):
    """Remove faint halo pixels and soften the edge slightly."""
    r, g, b, a = rgba.convert("RGBA").split()
    a = a.point(lambda v: 0 if v < 24 else v)
    a = a.filter(ImageFilter.MinFilter(3)).filter(ImageFilter.GaussianBlur(0.8))
    return Image.merge("RGBA", (r, g, b, a))


def place_product(product_rgba, size=1024, max_w=0.60, max_h=0.58, base_y=0.80):
    """Returns (canvas, mask, layer). Mask: white = area the model may paint."""
    product = product_rgba.convert("RGBA")
    bbox = product.getchannel("A").getbbox()
    if bbox is None:
        raise ValueError("No product found in the photo")
    product = product.crop(bbox)
    s = min(size * max_w / product.width, size * max_h / product.height)
    product = product.resize((max(1, round(product.width * s)), max(1, round(product.height * s))), Image.LANCZOS)
    x = (size - product.width) // 2
    y = round(size * base_y) - product.height
    layer = Image.new("RGBA", (size, size), (0, 0, 0, 0))
    layer.paste(product, (x, y))
    alpha = layer.getchannel("A")
    keep = alpha.point(lambda v: 255 if v > 200 else 0).filter(ImageFilter.MinFilter(7))   # shrink, so edges blend
    mask = ImageChops.invert(keep).filter(ImageFilter.GaussianBlur(4))
    canvas = Image.new("RGB", (size, size), (232, 232, 232))
    canvas.paste(product.convert("RGB"), (x, y), product.getchannel("A"))
    return canvas, mask, layer


def add_contact_shadow(img_rgb, layer, strength=0.38):
    alpha = layer.getchannel("A")
    bbox = alpha.getbbox()
    if bbox is None:
        return img_rgb
    l, t, r, b = bbox
    w, h = r - l, b - t
    sh = alpha.crop(bbox).transpose(Image.FLIP_TOP_BOTTOM).resize((w, max(8, h // 7)), Image.LANCZOS)
    shadow = Image.new("L", img_rgb.size, 0)
    shadow.paste(sh, (l, b - sh.height // 3))
    shadow = shadow.filter(ImageFilter.GaussianBlur(14)).point(lambda v: int(v * strength))
    return Image.composite(Image.new("RGB", img_rgb.size, (0, 0, 0)), img_rgb, shadow)

### 10.3 Photoshoot engine
Generates the scene first, places the real product in it, blends the lighting, then pastes the real product pixels back unchanged.

In [38]:
import random
if refine_pipe is None:
    raise RuntimeError("This version needs the RealVisXL model. Set ULTRA = True in the Configuration section and rerun from section 9.1.")

# same RealVisXL weights as refine_pipe, used as a text-to-image pipeline (no extra memory)
t2i_pipe = AutoPipelineForText2Image.from_pipe(refine_pipe)


def build_prompt(subject, scene):
    return (f"{scene}. A {subject} stands on the surface in the foreground, soft natural light, "
            "realistic contact shadow and reflections on the surface, blurred background with depth, "
            "shot on a full-frame camera, 85mm lens, high detail, photorealistic")


def make_scene(scene_prompt, gen, steps=30):
    prompt = (f"{scene_prompt}, empty surface in the lower foreground, eye-level view, nothing on the surface, "
              "no people, natural soft light, shallow depth of field, shot on a full-frame camera, "
              "85mm lens, photorealistic, high detail")
    return t2i_pipe(prompt=prompt, negative_prompt=NEGATIVE, num_inference_steps=steps,
                    guidance_scale=6.5, height=1024, width=1024, generator=gen).images[0]


def make_photoshoot(product_rgba, scene_prompt, subject="product", steps=30, seed=None, debug=False):
    gen = torch.Generator("cpu").manual_seed(seed if seed is not None else random.randint(0, 2**31 - 1))
    _, _, layer = place_product(clean_cutout(product_rgba))           # only the product layer is needed
    scene = make_scene(scene_prompt, gen, steps).convert("RGB")       # 1) the place

    comp = add_contact_shadow(scene, layer)                           # 2) product + shadow in the scene
    comp = comp.convert("RGBA")
    comp.alpha_composite(layer)
    comp = comp.convert("RGB")

    harmonized = refine_pipe(prompt=build_prompt(subject, scene_prompt), negative_prompt=NEGATIVE,
                             image=comp, strength=0.25, num_inference_steps=steps,
                             guidance_scale=6.0, generator=gen).images[0]   # 3) blend the lighting

    out = harmonized.convert("RGBA")
    out.alpha_composite(layer)                                        # 4) the REAL product pixels, unchanged
    out = out.convert("RGB")
    return (out, scene, comp, harmonized) if debug else out

print("Photoshoot engine ready.")

Photoshoot engine ready.


## 11. FastAPI Routes
All routes are registered here, before the server starts.

### 11.1 App and request models

In [39]:
import threading
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel

app = FastAPI(title="CampaignForge AI API")

gpu_lock = threading.Lock()          # one GPU job at a time


class CampaignRequest(BaseModel):
    product_name: str
    product_description: str
    target_audience: str
    campaign_goal: str
    brand_tone: str
    custom_tone: str | None = None
    offer_or_price: str | None = None
    key_differentiator: str | None = None
    direction_choice: int


class CampaignInput(BaseModel):
    product_name: str
    product_description: str
    target_audience: str
    campaign_goal: str
    brand_tone: str
    custom_tone: str | None = None
    offer_or_price: str | None = None
    key_differentiator: str | None = None


class ContentRequest(BaseModel):
    user_input: CampaignInput
    campaign_strategy: dict
    direction: dict


class ContentRequestL(ContentRequest):
    language: str = "English"              # "English" or "Arabic"

### 11.2 `GET /`, `GET /health`, `POST /generate-campaign`

In [40]:

@app.get("/")
def home():
    return FileResponse("/kaggle/working/index.html")


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/generate-campaign")
def generate_campaign_api(request: CampaignRequest):
    user_input = create_user_input({
        "product_name": request.product_name,
        "product_description": request.product_description,
        "target_audience": request.target_audience,
        "campaign_goal": request.campaign_goal,
        "brand_tone": request.brand_tone,
        "custom_tone": request.custom_tone,
        "offer_or_price": request.offer_or_price,
        "key_differentiator": request.key_differentiator
    })

    final_campaign = run_campaign(
        user_input,
        direction_choice=request.direction_choice
    )

    return final_campaign.model_dump()

### 11.3 `POST /directions`

In [41]:
@app.post("/directions")
def directions_api(req: CampaignInput):
    with gpu_lock:
        data = generate_campaign(create_user_input(req.model_dump()))
    return {
        "product_understanding": data["product_understanding"].model_dump(),
        "campaign_strategy": data["campaign_strategy"].model_dump(),
        "campaign_directions": data["campaign_directions"].model_dump(),
        "knowledge_used": data["knowledge_used"],
        "recommended": recommend_direction(req.campaign_goal),
    }

### 11.4 `POST /content` (English, or Arabic through Gemini)

In [42]:
@app.post("/content")
def content_api_lang(req: ContentRequestL):
    ui = create_user_input(req.user_input.model_dump())
    strategy = CampaignStrategy(**req.campaign_strategy)
    direction = CampaignDirection(**req.direction)
    wants_arabic = req.language.strip().lower().startswith("ar")

    with gpu_lock:                         # Mistral writes the English ad (rules and facts enforced here)
        platform = platform_chain.invoke({
            "product_name": ui.product_name, "product_description": ui.product_description,
            "target_audience": ui.target_audience, "campaign_goal": ui.campaign_goal.value,
            "brand_tone": ui.brand_tone.value, "offer_or_price": ui.offer_or_price or "Not provided",
            "key_differentiator": ui.key_differentiator or "Not provided",
            "core_message": strategy.core_message, "value_proposition": strategy.value_proposition,
            "emotional_angle": strategy.emotional_angle, "cta": strategy.cta,
            "content_pillars": strategy.content_pillars, "direction_name": direction.name,
            "strategic_angle": direction.strategic_angle, "key_message": direction.key_message,
            "recommended_cta": direction.recommended_cta, "content_approach": direction.content_approach,
        })

    instagram, tiktok, language, warnings = platform.instagram, platform.tiktok, "English", []
    if wants_arabic:
        facts = {"product_name": ui.product_name, "product_description": ui.product_description,
                 "offer_or_price": ui.offer_or_price or "Not provided",
                 "key_differentiator": ui.key_differentiator or "Not provided"}
        try:
            ar, issues = adapt_to_arabic(
                {"instagram": platform.instagram.model_dump(), "tiktok": platform.tiktok.model_dump()}, facts)
            instagram, tiktok, language, warnings = PlatformContent(**ar["instagram"]), PlatformContent(**ar["tiktok"]), "Arabic", issues
        except Exception as e:
            warnings = [f"Arabic adaptation failed, showing English: {str(e)[:120]}"]

    final = FinalCampaign(
        product_name=ui.product_name, campaign_goal=ui.campaign_goal.value, brand_tone=ui.brand_tone.value,
        campaign_strategy=strategy, selected_direction=direction, instagram=instagram, tiktok=tiktok)
    out = final.model_dump()
    out["language"], out["warnings"] = language, warnings
    return out

### 11.5 `POST /image` (product photo -> photoshoot, otherwise generic)

In [43]:
import base64, io


def backdrop_prompt(concept):
    """Turn the visual concept into a background-only scene (no people, no product)."""
    try:
        out = _gemini_json(
            "Rewrite this ad visual concept as a short BACKGROUND-ONLY scene for an image generator "
            "(max 25 words). Describe only the place, surface, lighting and mood. No people, no hands, "
            "no products, no text. Keep open empty space in the center foreground. "
            'Return JSON like {"scene": "..."}.\n\nConcept: ' + concept
        )
        scene = str(out["scene"]).strip()
        if scene:
            return scene
    except Exception:
        pass
    return "a clean modern desk by a window with soft daylight, blurred background"


@app.post("/image")
def image_api(req: ImageRequest):
    if req.product_photo:                                    # real product photo: realistic photoshoot
        photo = load_product_photo(req.product_photo)        # checks type, size and pixels
        cutout = remove(photo, session=rembg_session)
        if cutout.getchannel("A").getbbox() is None:
            raise HTTPException(400, "No product was found in the photo. Try a clearer photo of the product.")
        scene = backdrop_prompt(req.visual_concept)
        subject = " ".join(req.product_description.split()[:10]) or req.product_name
        with gpu_lock:
            img = make_photoshoot(cutout, scene, subject=subject)
        mode, prompt = "product_photo", scene
    else:                                                    # no photo: quick generic AI picture
        desc = " ".join(req.product_description.split()[:12])
        prompt = (f"professional advertising photo, {desc}, {req.visual_concept}, "
                  "soft studio lighting, sharp focus, photorealistic, no text, no logo")
        with gpu_lock:
            img = image_pipe(prompt=prompt, num_inference_steps=2, guidance_scale=0.0,
                             height=512, width=512).images[0]
        mode = "generic"

    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return {"image": "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode(),
            "prompt": prompt, "mode": mode}

## 12. Web Page
Writes `index.html` (served by `GET /`). It includes the Arabic language dropdown, the product-photo upload and the ad-photo button.

In [44]:
%%writefile index.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>CampaignForge AI</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap" rel="stylesheet">
<link href="https://fonts.googleapis.com/css2?family=Cairo:wght@400;600;700&display=swap" rel="stylesheet">
<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
<script src="https://cdn.tailwindcss.com"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/html2pdf.js/0.10.1/html2pdf.bundle.min.js"></script>
<style>
  body{font-family:'Inter',system-ui,sans-serif;background:#eef1f8;color:#1e2540}
  .side{background:radial-gradient(500px 300px at 0% 0%,rgba(109,93,246,.35),transparent 60%),#0e1430}
  .field{width:100%;background:rgba(255,255,255,.07);border:1px solid rgba(255,255,255,.12);border-radius:10px;padding:.55rem .75rem;font-size:.85rem;color:#fff;outline:none}
  .field::placeholder{color:rgba(255,255,255,.35)}
  .field:focus{border-color:#8b7dff;box-shadow:0 0 0 3px rgba(139,125,255,.25)}
  .field option{color:#111}
  .lbl{display:block;font-size:.75rem;font-weight:600;color:rgba(255,255,255,.7);margin:.8rem 0 .3rem}
  .card{background:#fff;border:1px solid #e3e7f2;border-radius:16px;box-shadow:0 1px 3px rgba(20,30,80,.05)}
  .chip{display:inline-block;background:#eeebff;color:#5340e0;border-radius:999px;padding:.15rem .7rem;margin:.15rem .25rem .15rem 0;font-size:.78rem;font-weight:500}
  .step{display:flex;align-items:center;gap:.5rem;font-size:.8rem;font-weight:600;color:#8a91ab}
  .step i{width:26px;height:26px;border-radius:50%;display:grid;place-items:center;background:#e3e7f2;font-style:normal;font-size:.75rem}
  .step.on{color:#1e2540}.step.on i{background:#6d5df6;color:#fff}
  .step.done i{background:#22c55e;color:#fff}
  .bar{height:4px;background:#e3e7f2;border-radius:4px;overflow:hidden}
  .bar span{display:block;height:100%;width:35%;background:#6d5df6;border-radius:4px;animation:slide 1.3s ease-in-out infinite}
  @keyframes slide{0%{margin-left:-35%}100%{margin-left:100%}}
  *:focus-visible{outline:2px solid #6d5df6;outline-offset:2px}
  @media (prefers-reduced-motion:reduce){.bar span{animation:none;width:100%}}
</style>
</head>
<body class="min-h-screen lg:h-screen flex flex-col lg:flex-row">

<!-- Sidebar: brief form -->
<aside class="side text-white w-full lg:w-[340px] shrink-0 flex flex-col lg:overflow-y-auto p-5">
  <div class="flex items-center gap-3 mb-2">
    <div class="w-10 h-10 rounded-xl grid place-items-center" style="background:linear-gradient(135deg,#6d5df6,#ff7a59)"><i class="fa-solid fa-bolt"></i></div>
    <div><div class="font-bold text-lg leading-tight">CampaignForge AI</div><div class="text-xs text-white/50">Product brief to full campaign</div></div>
  </div>

  <label class="lbl" for="name">Product name *</label>
  <input id="name" class="field" placeholder="Smart Water Bottle">
  <label class="lbl" for="desc">Product description *</label>
  <textarea id="desc" rows="3" class="field resize-none" placeholder="What it is and what it does"></textarea>
  <label class="lbl" for="aud">Target audience *</label>
  <input id="aud" class="field" placeholder="Busy university students">
  <div class="grid grid-cols-2 gap-3">
    <div><label class="lbl" for="goal">Campaign goal *</label>
      <select id="goal" class="field"><option>Awareness</option><option>Engagement</option><option selected>Sales</option><option>Launch</option></select></div>
    <div><label class="lbl" for="tone">Brand tone *</label>
      <select id="tone" class="field"><option>Professional</option><option>Playful</option><option>Luxury</option><option>Bold</option><option>Friendly</option><option>Custom</option></select></div>
  </div>
  <div id="customWrap" class="hidden"><label class="lbl" for="custom">Describe your tone *</label><input id="custom" class="field" placeholder="Warm and witty"></div>
  <label class="lbl" for="lang">Ad language</label>
  <select id="lang" class="field"><option value="English">English</option><option value="Arabic">العربية (Arabic)</option></select>
  <label class="lbl" for="offer">Offer or price <span class="text-white/40 font-normal">(optional)</span></label>
  <input id="offer" class="field" placeholder="20% off">
  <label class="lbl" for="diff">Key differentiator <span class="text-white/40 font-normal">(optional)</span></label>
  <input id="diff" class="field" placeholder="What makes it different">

  <label class="lbl" for="photo">Product photo <span class="text-white/40 font-normal">(optional)</span></label>
  <input id="photo" type="file" accept="image/png,image/jpeg,image/webp" class="field">
  <p id="photoMsg" class="text-xs text-white/60 mt-1"></p>
  <img id="photoPrev" alt="Product photo preview" class="hidden mt-2 rounded-lg max-h-24">
  <p id="formErr" class="text-sm text-red-300 mt-3 hidden"></p>
  <button id="gen" class="mt-5 w-full py-3 rounded-xl font-semibold text-white disabled:opacity-50" style="background:linear-gradient(135deg,#6d5df6,#8b5cf6)">
    <i class="fa-solid fa-wand-magic-sparkles mr-2"></i>Generate strategy</button>

  <div class="mt-6">
    <button id="newBtn" class="w-full py-2 rounded-xl border border-white/20 text-sm text-white/80 hover:bg-white/10"><i class="fa-solid fa-plus mr-2"></i>New campaign</button>
    <div class="flex items-center justify-between mt-4 mb-2"><span class="text-xs font-bold text-white/50">RECENT CAMPAIGNS</span><button id="clearHist" class="text-xs text-white/40 hover:text-white">Clear</button></div>
    <div id="hist" class="space-y-1"></div>
  </div>

  <div class="mt-auto pt-6 flex items-center gap-2 text-xs text-white/60">
    <span id="dot" class="w-2 h-2 rounded-full bg-amber-400"></span><span id="statusTxt">Checking server</span>
  </div>
</aside>

<!-- Workspace -->
<main class="flex-1 min-w-0 flex flex-col lg:overflow-hidden">
  <header class="px-6 py-4 flex flex-wrap items-center gap-x-8 gap-y-2 border-b border-slate-200 bg-white/70">
    <h1 class="font-bold text-lg mr-2">Workspace</h1>
    <div id="s1" class="step on"><i>1</i>Brief</div>
    <div id="s2" class="step"><i>2</i>Strategy and direction</div>
    <div id="s3" class="step"><i>3</i>Final campaign</div>
  </header>
  <div id="view" class="flex-1 lg:overflow-y-auto p-6"></div>
</main>

<script>
const $ = s => document.querySelector(s);
const esc = t => String(t ?? "").replace(/[&<>"']/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"}[c]));
const chips = a => (a||[]).map(x => `<span class="chip">${esc(x)}</span>`).join("");
const list = a => `<ul class="list-disc pl-5 space-y-1 text-sm">${(a||[]).map(x=>`<li>${esc(x)}</li>`).join("")}</ul>`;
const view = $("#view");
let S = {inputs:null, data:null, final:null}, copies = [], timer = null;
let PHOTO = null;   // real product photo as a data URL
const PHOTO_TYPES = ["image/png","image/jpeg","image/webp"], PHOTO_MAX = 5*1024*1024;
function clearPhoto(){ PHOTO = null; $("#photo").value = ""; $("#photoMsg").textContent = ""; $("#photoPrev").classList.add("hidden"); }

/* ---------- campaign history (browser localStorage) ---------- */
const HK = "campaignforge.history.v1";
const loadHist = () => { try { return JSON.parse(localStorage.getItem(HK)) || []; } catch(e){ return []; } };
const saveHist = h => { try { localStorage.setItem(HK, JSON.stringify(h.slice(0,10))); } catch(e){} };
function remember(){
  if(!S.data) return;
  if(!S.id) S.id = Date.now().toString(36);
  const h = loadHist().filter(x => x.id !== S.id);
  h.unshift({id:S.id, name:S.inputs.product_name, inputs:S.inputs, data:S.data, final:S.final});
  saveHist(h); renderHist();
}
function renderHist(){
  const h = loadHist();
  $("#hist").innerHTML = h.length ? h.map(x => `<div class="flex items-center gap-1">
    <button data-h="${x.id}" class="flex-1 text-left text-sm text-white/80 hover:bg-white/10 rounded-lg px-2 py-1.5 truncate"><i class="fa-regular fa-clock mr-2 text-white/40"></i>${esc(x.name)}${x.final ? "" : ' <span class="text-white/40 text-xs">(draft)</span>'}</button>
    <button data-hd="${x.id}" aria-label="Delete campaign" class="text-white/30 hover:text-red-300 px-1"><i class="fa-solid fa-xmark"></i></button></div>`).join("")
    : '<p class="text-xs text-white/40">No campaigns yet.</p>';
}
function fillForm(i){
  const m = {name:"product_name", desc:"product_description", aud:"target_audience", custom:"custom_tone", offer:"offer_or_price", diff:"key_differentiator"};
  for(const id in m) $("#"+id).value = i[m[id]] || "";
  $("#goal").value = i.campaign_goal; $("#tone").value = i.brand_tone; $("#lang").value = i.language || "English";
  $("#customWrap").classList.toggle("hidden", i.brand_tone !== "Custom");
}

/* ---------- campaign intelligence flow ---------- */
function flow(dir, done){
  const u = S.data.product_understanding, s = S.data.campaign_strategy;
  const steps = [
    ["fa-box","Product", S.inputs.product_name],
    ["fa-magnifying-glass-chart","Understanding", u.product_positioning],
    ["fa-user","Customer need", `${(u.pain_points||[])[0]||""} \u2192 ${(u.customer_desires||[])[0]||""}`],
    ["fa-chess","Strategy", s.emotional_angle],
    ["fa-compass","Direction", dir ? `${dir.name}: ${dir.strategic_angle}` : "Choose one below"],
    ["fa-pen-nib","Content", done ? "Instagram and TikTok content ready" : "Created after you choose"]
  ];
  return `<section class="card p-5"><h2 class="font-bold mb-3"><i class="fa-solid fa-diagram-project text-[#6d5df6] mr-2"></i>Campaign intelligence</h2>
    <div class="grid md:grid-cols-3 xl:grid-cols-6 gap-3">${steps.map((x,i) => `<div class="relative rounded-xl border border-slate-200 p-3 ${i>3&&!(i===4?dir:done)?"bg-slate-50 opacity-70":"bg-white"}">
      <div class="text-xs font-bold text-[#6d5df6] mb-1"><i class="fa-solid ${x[0]} mr-1"></i>${x[1]}</div><p class="text-xs text-slate-600">${esc(x[2])}</p>
      ${i<5 ? '<i class="fa-solid fa-chevron-right hidden xl:block absolute -right-3 top-1/2 text-slate-300 z-10"></i>' : ""}</div>`).join("")}</div></section>`;
}

function setStage(n){
  [1,2,3].forEach(i => { const e = $("#s"+i); e.className = "step" + (i<n?" done":i===n?" on":""); e.firstChild.textContent = i<n ? "✓" : i; });
}
async function post(path, body){
  const r = await fetch(path, {method:"POST", headers:{"Content-Type":"application/json","ngrok-skip-browser-warning":"1"}, body:JSON.stringify(body)});
  if(!r.ok) throw new Error("Server error " + r.status + ". Check the notebook logs.");
  return r.json();
}

/* ---------- views ---------- */
function showEmpty(){
  view.innerHTML = `<div class="h-full grid place-items-center text-center"><div class="max-w-md">
    <div class="mx-auto w-16 h-16 rounded-2xl grid place-items-center text-2xl text-white mb-4" style="background:linear-gradient(135deg,#6d5df6,#ff7a59)"><i class="fa-solid fa-rocket"></i></div>
    <h2 class="text-2xl font-bold mb-2">Start with a product brief</h2>
    <p class="text-slate-500 text-sm">Fill in the form on the left. You will get a strategy, three directions, and ready-to-post Instagram and TikTok content.</p></div></div>`;
}
function showLoading(msgs){
  let i = 0;
  view.innerHTML = `<div class="h-full grid place-items-center"><div class="card p-8 w-full max-w-md text-center">
    <div class="bar mb-5"><span></span></div><p id="lmsg" class="font-medium">${esc(msgs[0])}</p>
    <p class="text-xs text-slate-400 mt-2">This can take around a minute.</p></div></div>`;
  clearInterval(timer);
  timer = setInterval(() => { i = Math.min(i+1, msgs.length-1); const e=$("#lmsg"); if(e) e.textContent = msgs[i]; }, 15000);
}
function showError(msg){
  clearInterval(timer);
  view.innerHTML = `<div class="card p-6 max-w-lg mx-auto text-center"><i class="fa-solid fa-triangle-exclamation text-amber-500 text-2xl mb-3"></i>
    <p class="font-semibold mb-1">Something went wrong</p><p class="text-sm text-slate-500 mb-4">${esc(msg)}</p>
    <button data-act="${S.data?'dirs':'reset'}" class="px-4 py-2 rounded-lg bg-slate-900 text-white text-sm">Go back</button></div>`;
}

const DIR_META = {
  product_focused:    {icon:"fa-cube",     color:"#2563eb", bg:"#eff6ff", label:"Product",    focus:"The product and what it does", best:"Launch and education"},
  lifestyle_focused:  {icon:"fa-heart",    color:"#db2777", bg:"#fdf2f8", label:"Lifestyle",  focus:"The customer's daily life",    best:"Awareness and engagement"},
  conversion_focused: {icon:"fa-bullseye", color:"#d97706", bg:"#fffbeb", label:"Conversion", focus:"The offer and the goal",       best:"Sales"}
};
function showDirections(){
  clearInterval(timer); setStage(2);
  const {product_understanding:u, campaign_strategy:s, campaign_directions:d, knowledge_used:kn=[], recommended:rec={}} = S.data;
  const keys = Object.keys(DIR_META);
  const cmpRows = [["Main focus",k=>DIR_META[k].focus],["Best for",k=>DIR_META[k].best],["Strategic angle",k=>d[k].strategic_angle],["Call to action",k=>d[k].recommended_cta]];
  const cmp = `<div class="card overflow-x-auto mb-5"><table class="w-full text-sm"><thead><tr><th class="p-3"></th>${keys.map(k=>`<th class="p-3 text-left" style="color:${DIR_META[k].color}">${DIR_META[k].label}</th>`).join("")}</tr></thead>
    <tbody>${cmpRows.map(r=>`<tr class="border-t border-slate-100"><td class="p-3 text-xs font-bold text-slate-400 whitespace-nowrap">${r[0]}</td>${keys.map(k=>`<td class="p-3 align-top">${esc(r[1](k))}</td>`).join("")}</tr>`).join("")}</tbody></table></div>
    ${rec.reason ? `<p class="text-xs text-slate-500 mb-4"><i class="fa-solid fa-circle-info mr-1"></i>${esc(rec.reason)}</p>` : ""}`;
  const know = `<section class="card p-5"><h2 class="font-bold mb-3"><i class="fa-solid fa-book-open text-[#6d5df6] mr-2"></i>Marketing knowledge used</h2>${kn.length
    ? `<ul class="text-sm space-y-1 mb-3">${kn.map(x=>`<li><i class="fa-solid fa-check text-emerald-500 mr-2"></i>${esc(x.section)}${x.page?` <span class="text-slate-400">(page ${x.page})</span>`:""}</li>`).join("")}</ul>
       ${kn.slice(0,2).map(x=>`<blockquote class="border-l-4 border-[#6d5df6]/40 bg-slate-50 rounded-r-lg p-3 text-xs text-slate-600 mb-2">${esc(x.excerpt)}</blockquote>`).join("")}`
    : `<p class="text-sm text-slate-400">No sources were saved for this campaign.</p>`}</section>`;
  view.innerHTML = `<div class="max-w-6xl mx-auto space-y-6">
    ${flow(null, false)}
    <section class="card p-5"><h2 class="font-bold mb-3"><i class="fa-solid fa-magnifying-glass-chart text-[#6d5df6] mr-2"></i>Product understanding</h2>
      <p class="text-sm mb-4 text-slate-600">${esc(u.product_positioning)}</p>
      <div class="grid md:grid-cols-2 xl:grid-cols-4 gap-5">
        <div><h3 class="text-xs font-bold text-slate-400 mb-2">Features</h3>${list(u.features)}</div>
        <div><h3 class="text-xs font-bold text-slate-400 mb-2">Benefits</h3>${list(u.benefits)}</div>
        <div><h3 class="text-xs font-bold text-slate-400 mb-2">Pain points</h3>${list(u.pain_points)}</div>
        <div><h3 class="text-xs font-bold text-slate-400 mb-2">Customer desires</h3>${list(u.customer_desires)}</div></div></section>
    <section class="card p-5"><h2 class="font-bold mb-3"><i class="fa-solid fa-chess text-[#6d5df6] mr-2"></i>Campaign strategy</h2>
      <div class="grid md:grid-cols-2 gap-4 text-sm">
        <p><b>Core message</b><br>${esc(s.core_message)}</p><p><b>Value proposition</b><br>${esc(s.value_proposition)}</p>
        <p><b>Emotional angle</b><br>${esc(s.emotional_angle)}</p><p><b>Call to action</b><br>${esc(s.cta)}</p></div>
      <div class="mt-3">${chips(s.content_pillars)}</div></section>
    ${know}
    <section><h2 class="font-bold mb-3">Compare directions</h2>${cmp}<div class="grid lg:grid-cols-3 gap-5">
      ${Object.keys(DIR_META).map(k => { const x=d[k], m=DIR_META[k]; return `
        <div class="card p-5 flex flex-col ${k===rec.key?"ring-2 ring-[#6d5df6]":""}">
          ${k===rec.key?'<span class="self-start text-[10px] font-bold tracking-wide text-white bg-[#6d5df6] rounded-full px-2.5 py-1 mb-3">RECOMMENDED FOR YOUR GOAL</span>':""}
          <div class="w-10 h-10 rounded-xl grid place-items-center mb-3" style="background:${m.bg};color:${m.color}"><i class="fa-solid ${m.icon}"></i></div>
          <h3 class="font-bold">${esc(x.name)}</h3><p class="text-xs text-slate-500 mb-3">${esc(x.strategic_angle)}</p>
          <p class="text-sm mb-3"><b>Message</b><br>${esc(x.key_message)}</p>
          <p class="text-sm mb-3"><b>CTA</b><br>${esc(x.recommended_cta)}</p>
          <p class="text-sm mb-4"><b>Approach</b><br>${esc(x.content_approach)}</p>
          <button data-dir="${k}" class="mt-auto py-2.5 rounded-xl text-white text-sm font-semibold" style="background:${m.color}">Use this direction</button></div>`; }).join("")}
    </div></section></div>`;
}

function platformCard(key, p){
  const ig = key === "instagram", ar = !!(S.final && S.final.language === "Arabic");
  const AR = ar ? 'dir="rtl" style="font-family:Cairo,Inter,sans-serif"' : "";
  const idx = copies.push(p.caption) - 1;
  return `<section class="card p-5">
    <div class="flex items-center gap-3 mb-4"><div class="w-10 h-10 rounded-xl grid place-items-center text-white" style="background:${ig?"linear-gradient(135deg,#f58529,#dd2a7b,#8134af)":"#111"}">
      <i class="fa-brands ${ig?"fa-instagram":"fa-tiktok"}"></i></div><h3 class="font-bold">${ig?"Instagram":"TikTok"}</h3></div>
    <p class="text-lg font-semibold mb-3" ${AR}>${esc(p.hook)}</p>
    <div class="relative bg-slate-50 rounded-xl p-4 text-sm whitespace-pre-wrap" ${AR}>${esc(p.caption)}
      <button data-copy="${idx}" class="absolute top-2 ${ar?"left-2":"right-2"} text-xs px-2 py-1 rounded-md bg-white border border-slate-200 hover:bg-slate-100" data-html2canvas-ignore><i class="fa-regular fa-copy mr-1"></i>Copy</button></div>
    <p class="text-sm mt-3" ${AR}><b>CTA:</b> ${esc(p.cta)}</p>
    <div class="mt-2" ${AR}>${chips(p.hashtags)}</div>
    <div class="mt-3 border border-dashed border-slate-300 rounded-xl p-3 text-sm"><b><i class="fa-regular fa-image mr-1"></i>Visual concept</b><br>${esc(p.visual_concept)}</div>
    <button data-adimg="${ig?"instagram":"tiktok"}" data-html2canvas-ignore class="mt-3 w-full py-2 rounded-xl bg-[#6d5df6] text-white text-sm font-semibold hover:opacity-90"><i class="fa-solid fa-camera mr-2"></i>Generate ad photo</button>
    <div id="adimg-${ig?"instagram":"tiktok"}"></div></section>`;
}
function showFinal(){
  clearInterval(timer); setStage(3); copies = [];
  const c = S.final;
  view.innerHTML = `<div class="max-w-6xl mx-auto">
    <div class="flex flex-wrap items-center gap-3 mb-5">
      <div class="mr-auto"><h2 class="text-xl font-bold">${esc(c.product_name)}</h2>
        <p class="text-sm text-slate-500">${esc(c.campaign_goal)} | ${esc(c.brand_tone)} tone | ${esc(c.selected_direction.name)}${c.language === "Arabic" ? " | Arabic" : ""}</p>${(c.warnings||[]).length ? '<p class="text-xs text-amber-600 mt-1">' + esc(c.warnings.join("; ")) + '</p>' : ""}</div>
      <button data-act="pdf" class="px-3 py-2 rounded-lg bg-white border border-slate-200 text-sm"><i class="fa-solid fa-file-pdf mr-1.5"></i>Export PDF</button>
      <button data-act="json" class="px-3 py-2 rounded-lg bg-white border border-slate-200 text-sm"><i class="fa-solid fa-download mr-1.5"></i>JSON</button>
      <button data-act="dirs" class="px-3 py-2 rounded-lg bg-white border border-slate-200 text-sm">Other direction</button>
      <button data-act="reset" class="px-3 py-2 rounded-lg bg-slate-900 text-white text-sm">New campaign</button></div>
    <div id="report" class="space-y-5">${flow(c.selected_direction, true)}
      <section class="card p-5 text-sm"><p><b>Core message:</b> ${esc(c.campaign_strategy.core_message)}</p>
        <p class="mt-1"><b>Direction message:</b> ${esc(c.selected_direction.key_message)}</p></section>
      <div class="grid lg:grid-cols-2 gap-5">${platformCard("instagram", c.instagram)}${platformCard("tiktok", c.tiktok)}</div></div></div>`;
}

/* ---------- actions ---------- */
async function generate(){
  const v = id => $("#"+id).value.trim(), tone = $("#tone").value;
  const req = {"Product name":v("name"), "Description":v("desc"), "Audience":v("aud")};
  if(tone === "Custom") req["Custom tone"] = v("custom");
  const missing = Object.keys(req).filter(k => !req[k]), err = $("#formErr");
  if(missing.length){ err.textContent = "Fill in: " + missing.join(", "); err.classList.remove("hidden"); return; }
  err.classList.add("hidden");
  S.inputs = {product_name:v("name"), product_description:v("desc"), target_audience:v("aud"), campaign_goal:$("#goal").value, brand_tone:tone,
              custom_tone:v("custom")||null, offer_or_price:v("offer")||null, key_differentiator:v("diff")||null, language:$("#lang").value};
  S.id = null; S.data = null; S.final = null; setStage(1); $("#gen").disabled = true;
  showLoading(["Analyzing your product","Retrieving marketing knowledge","Building the campaign strategy","Drafting three directions"]);
  try{ S.data = await post("/directions", S.inputs); remember(); showDirections(); }
  catch(e){ showError(e.message); }
  $("#gen").disabled = false;
}
async function pick(key){
  showLoading(["Writing Instagram content","Writing TikTok content"]);
  try{
    S.final = await post("/content", {user_input:S.inputs, campaign_strategy:S.data.campaign_strategy, direction:S.data.campaign_directions[key], language:S.inputs.language||"English"});
    remember(); showFinal();
  }catch(e){ showError(e.message); }
}
view.addEventListener("click", e => {
  const b = e.target.closest("button"); if(!b) return;
  if(b.dataset.dir) return pick(b.dataset.dir);
  if(b.dataset.copy){ navigator.clipboard.writeText(copies[b.dataset.copy]); b.innerHTML = '<i class="fa-solid fa-check mr-1"></i>Copied'; return; }
  const a = b.dataset.act;
  if(a === "dirs") showDirections();
  if(a === "reset"){ S = {inputs:null,data:null,final:null}; clearPhoto(); setStage(1); showEmpty(); }
  if(a === "pdf") html2pdf().set({margin:10, filename:"campaign.pdf", jsPDF:{format:"a4"}}).from($("#report")).save();
  if(a === "json"){ const l=document.createElement("a"); l.href=URL.createObjectURL(new Blob([JSON.stringify(S.final,null,2)],{type:"application/json"})); l.download="campaign.json"; l.click(); }
});
$("#hist").onclick = e => {
  const d = e.target.closest("[data-hd]"), o = e.target.closest("[data-h]");
  if(d){ saveHist(loadHist().filter(x => x.id !== d.dataset.hd)); renderHist(); return; }
  if(o){ const x = loadHist().find(y => y.id === o.dataset.h); if(!x) return;
    S = {id:x.id, inputs:x.inputs, data:x.data, final:x.final}; clearPhoto(); fillForm(x.inputs); x.final ? showFinal() : showDirections(); }
};
$("#clearHist").onclick = () => { saveHist([]); renderHist(); };
$("#newBtn").onclick = () => { S = {inputs:null,data:null,final:null}; clearPhoto(); setStage(1); showEmpty(); };
$("#gen").onclick = generate;
$("#photo").onchange = e => {
  const f = e.target.files[0], msg = $("#photoMsg");
  if(!f){ clearPhoto(); return; }
  if(!PHOTO_TYPES.includes(f.type)){ clearPhoto(); msg.textContent = "Use a PNG, JPG or WEBP image."; return; }
  if(f.size > PHOTO_MAX){ clearPhoto(); msg.textContent = "Photo is too large. Maximum size is 5 MB."; return; }
  const rd = new FileReader();
  rd.onload = () => { PHOTO = rd.result; msg.textContent = f.name; const pv = $("#photoPrev"); pv.src = PHOTO; pv.classList.remove("hidden"); };
  rd.onerror = () => { clearPhoto(); msg.textContent = "Could not read the photo."; };
  rd.readAsDataURL(f);
};
$("#tone").onchange = e => $("#customWrap").classList.toggle("hidden", e.target.value !== "Custom");

fetch("/health").then(r => r.ok ? r.json() : Promise.reject()).then(() => {
  $("#dot").className = "w-2 h-2 rounded-full bg-emerald-400"; $("#statusTxt").textContent = "Model server online";
}).catch(() => { $("#statusTxt").textContent = "Server not reachable"; });

document.addEventListener("click", async e => {
  const b = e.target.closest("[data-adimg]"); if(!b) return;
  const p = b.dataset.adimg, box = document.getElementById("adimg-" + p);
  b.disabled = true; b.innerHTML = '<i class="fa-solid fa-spinner fa-spin mr-2"></i>Creating photo...'; box.innerHTML = "";
  try {
    const r = await fetch("/image", {method:"POST", headers:{"Content-Type":"application/json"},
      body: JSON.stringify({product_name:S.inputs.product_name, product_description:S.inputs.product_description,
                            visual_concept:S.final[p].visual_concept, platform:p, product_photo:PHOTO})});
    if(!r.ok){ const t = await r.text(); let m = t.slice(0,200); try{ const d = JSON.parse(t).detail; if(typeof d === "string") m = d; }catch(_){} throw new Error(m); }
    const j = await r.json();
    box.innerHTML = `<img src="${j.image}" alt="AI ad photo" class="rounded-xl w-full mt-3"><a download="${S.inputs.product_name}-${p}.png" href="${j.image}" data-html2canvas-ignore class="inline-block mt-2 text-sm font-semibold text-[#6d5df6]"><i class="fa-solid fa-download mr-1"></i>Download photo</a>`;
  } catch(err){ box.innerHTML = `<p class="text-sm text-red-500 mt-2">${esc(err.message)}</p>`; }
  b.disabled = false; b.innerHTML = '<i class="fa-solid fa-camera mr-2"></i>Regenerate ad photo';
});
renderHist();
showEmpty();
</script>
</body>
</html>

Writing index.html


In [45]:
import shutil

shutil.copy(
    "/kaggle/working/index.html",
    "/kaggle/working/CampaignForge_final_index.html"
)

print("Created: /kaggle/working/CampaignForge_final_index.html")

Created: /kaggle/working/CampaignForge_final_index.html


## 13. Pre-flight Check
Confirms every route and dependency exists before the server starts.

In [46]:
route_paths = {getattr(r, "path", "") for r in app.routes}
required_routes = ["/", "/health", "/directions", "/content", "/image", "/generate-campaign"]
print("routes:", sorted(route_paths))
missing_routes = [p for p in required_routes if p not in route_paths]
assert not missing_routes, f"Missing routes: {missing_routes}"

required_names = ["app", "gpu_lock", "image_pipe", "refine_pipe", "t2i_pipe", "ImageRequest",
                  "load_product_photo", "_gemini_json", "adapt_to_arabic", "rembg_session",
                  "make_photoshoot", "platform_chain", "directions_chain", "product_chain",
                  "vectordb", "generate_response"]
missing_names = [n for n in required_names if n not in globals()]
assert not missing_names, f"Missing objects: {missing_names}"
print("All routes registered and dependencies loaded.")

routes: ['/', '/content', '/directions', '/docs', '/docs/oauth2-redirect', '/generate-campaign', '/health', '/image', '/openapi.json', '/redoc']
All routes registered and dependencies loaded.


## 14. Ngrok & Server Startup

In [49]:
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

ngrok_token = UserSecretsClient().get_secret("NGROK_AUTH_TOKEN")
ngrok.set_auth_token(ngrok_token)

print("ngrok authentication configured successfully.")

ngrok authentication configured successfully.                                                       


In [50]:
import uvicorn

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=PORT)

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

print(f"FastAPI server started on port {PORT}.")

FastAPI server started on port 8000.


INFO:     Started server process [203]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [51]:
ngrok.kill()                       # closes any old tunnel from a previous run
public_url = ngrok.connect(PORT)

# The URL changes every time the runtime restarts. Copy the new one into the app settings.
API_URL = getattr(public_url, "public_url", None) or str(public_url).replace("NgrokTunnel: ", "").split('"')[1]

print("CampaignForge API URL:")
print(API_URL)

CampaignForge API URL:
https://subjudicial-dubitatively-dalila.ngrok-free.dev


## 15. Health Check
Quick check through the public tunnel. It does not use the GPU.

In [52]:
import requests, time

time.sleep(2)                      # give uvicorn a moment to bind
local = requests.get(f"http://localhost:{PORT}/health", timeout=15)
print("local  /health:", local.status_code, local.json())

public = requests.get(f"{API_URL}/health", timeout=30, headers={"ngrok-skip-browser-warning": "1"})
print("public /health:", public.status_code, public.json())

INFO:     127.0.0.1:54834 - "GET /health HTTP/1.1" 200 OK
local  /health: 200 {'status': 'ok'}
INFO:     34.148.193.192:0 - "GET /health HTTP/1.1" 200 OK
public /health: 200 {'status': 'ok'}


## 16. Optional: Full Text Test
Set `RUN_TEXT_TEST = True` to run a real `/directions` call through the tunnel (uses the GPU, takes a while). Leave it `False` for a normal *Run All*.

In [53]:
RUN_TEXT_TEST = False

if RUN_TEXT_TEST:
    test_payload = {
        "product_name": "AI Study Planner",
        "product_description": "An AI-powered planner that helps students organize their study tasks and schedule.",
        "target_audience": "University students",
        "campaign_goal": "Awareness",
        "brand_tone": "Friendly",
        "custom_tone": None,
        "offer_or_price": None,
        "key_differentiator": "Automatically organizes study tasks into a practical schedule."
    }
    r = requests.post(f"{API_URL}/directions", json=test_payload, timeout=600,
                      headers={"ngrok-skip-browser-warning": "1"})
    print("Status code:", r.status_code)
    print(r.json() if r.status_code == 200 else r.text)

INFO:     41.238.212.179:0 - "GET / HTTP/1.1" 200 OK
INFO:     41.238.212.179:0 - "GET /health HTTP/1.1" 200 OK
INFO:     41.238.212.179:0 - "POST /directions HTTP/1.1" 200 OK
INFO:     41.238.212.179:0 - "POST /content HTTP/1.1" 200 OK


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/7 [00:00<?, ?it/s]

INFO:     41.238.212.179:0 - "POST /image HTTP/1.1" 200 OK


  0%|          | 0/30 [00:00<?, ?it/s]

...

INFO:     41.238.212.179:0 - "POST /image HTTP/1.1" 200 OK


  0%|          | 0/30 [00:00<?, ?it/s]

The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['8 5 mm lens , high detail , photorealistic']
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['8 5 mm lens , high detail , photorealistic']


  0%|          | 0/7 [00:00<?, ?it/s]

INFO:     41.238.212.179:0 - "POST /image HTTP/1.1" 200 OK


**The backend is now live.** Keep this notebook session running. If Kaggle restarts the runtime, *Run All* again and use the new ngrok URL.

In [54]:
import requests
requests.get("http://127.0.0.1:8000/health").json()

INFO:     127.0.0.1:54490 - "GET /health HTTP/1.1" 200 OK


{'status': 'ok'}

INFO:     41.238.212.179:0 - "GET /health HTTP/1.1" 200 OK
